In [3]:
!hdfs dfs -mkdir -p /user/vey/ecommerce/raw
!hdfs dfs -mkdir -p /user/vey/ecommerce/processed

In [4]:
!hdfs dfs -ls -R /user/vey/ecommerce

drwxr-xr-x   - vey supergroup          0 2026-09-08 20:56 /user/vey/ecommerce/processed
drwxr-xr-x   - vey supergroup          0 2026-09-08 20:56 /user/vey/ecommerce/raw


In [21]:
!hdfs dfs -put -f transaksi_magelang.csv /user/vey/ecommerce/raw/
!hdfs dfs -put -f transaksi_yogyakarta.csv /user/vey/ecommerce/raw/
!hdfs dfs -put -f transaksi_semarang.csv /user/vey/ecommerce/raw/

In [22]:
!hdfs dfs -ls -h /user/vey/ecommerce/raw/

Found 3 items
-rw-r--r--   1 vey supergroup     12.0 K 2026-09-08 21:31 /user/vey/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 vey supergroup     11.9 K 2026-09-08 21:31 /user/vey/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 vey supergroup     12.4 K 2026-09-08 21:31 /user/vey/ecommerce/raw/transaksi_yogyakarta.csv


In [24]:
!hdfs dfs -get /user/vey/ecommerce/raw/transaksi_magelang.csv magelang_dari_hdfs.csv
!hdfs dfs -get /user/vey/ecommerce/raw/transaksi_semarang.csv semarang_dari_hdfs.csv
!hdfs dfs -get /user/vey/ecommerce/raw/transaksi_yogyakarta.csv yogyakarta_dari_hdfs.csv

In [26]:
import pandas as pd 

df_magelang = pd.read_csv("magelang_dari_hdfs.csv")
df_semarang = pd.read_csv("semarang_dari_hdfs.csv")
df_yogyakarta = pd.read_csv("yogyakarta_dari_hdfs.csv")

# Gabungkan ketiga DataFrame
df_gabungan = pd.concat([df_magelang, df_semarang, df_yogyakarta],  ignore_index=True)
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
595,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
596,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
597,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
598,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta


In [27]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Semarang      200
Yogyakarta    200
Name: count, dtype: int64

In [28]:
# menambahkan kolom pendapatan 
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [30]:
# tabel ringkasan
ringkasan_kota_kategori = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()
ringkasan_kota_kategori

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [36]:
# Menyimpan dua berkas di disk lokal

df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)

print("Kedua berkas berhasil disimpan di disk lokal.")

Kedua berkas berhasil disimpan di disk lokal.


In [38]:
# Mengunggah kedua berkas hasil olahan

!hdfs dfs -put data_gabungan_bersih.csv /user/vey/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/vey/ecommerce/processed/

# Membuktikan kedua berkas berhasil terunggah
!hdfs dfs -ls /user/vey/ecommerce/processed/

Found 2 items
-rw-r--r--   1 vey supergroup      41257 2026-09-09 04:06 /user/vey/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 vey supergroup        530 2026-09-09 04:06 /user/vey/ecommerce/processed/ringkasan_kota_kategori.csv


In [ ]:
# Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semua bercampur dalam satu folder

Kenutungan menyimpan data raw dan processed secara terpisah pada HDFS  membuat datnya lebih mudah diakses dan rapi. Data raw merupakan 
data asli yang belum diolah (data mentah), sedangkan data processing merupakan data yang sudah diolah dan siap digunakan. Dengan memisahkan
dua data tersebut, kita jadi lebih mudahdalam membedakan data asli maupun data hasil pengolahan. Jika suatu saat terjadi kesalahan dalam 
pengolahan data, data raw masih bisa digunkana untuk mengulang proses dari awal. Selain itu, pemisahan folder juga membuat kita lebih mudah 
mencari data yang dibutuhkan dan mengurangi kemungkinan salah menggunakan file. Kalau semua data disimpan dalam satu folder, file bisa
tercampur dan lebih sulit untuk dikelola. 

In [42]:
# mengecek ukuran total disk
!hdfs dfs -du -h /user/vey/ecommerce

# menghitung jumlah file & folder di dalamnya
!hdfs  -count /user/vey/ecommerce

40.8 K  40.8 K  /user/vey/ecommerce/processed
36.3 K  36.3 K  /user/vey/ecommerce/raw
/home/vey/hadoop/libexec/hadoop-functions.sh: line 2417: HDFS_-COUNT_USER: invalid variable name
ERROR: -count is not COMMAND nor fully qualified CLASSNAME.
Usage: hdfs [OPTIONS] SUBCOMMAND [SUBCOMMAND OPTIONS]

  OPTIONS is none or any of:

--buildpaths                       attempt to add class files from build tree
--config dir                       Hadoop config directory
--daemon (start|status|stop)       operate on a daemon
--debug                            turn on shell script debug mode
--help                             usage information
--hostnames list[,of,host,names]   hosts to use in worker mode
--hosts filename                   list of hosts to use in worker mode
--loglevel level                   set the log4j level for this command
--workers                          turn on worker mode

  SUBCOMMAND is one of:


    Admin Commands:

cacheadmin           configure the HDFS cache
crypt

In [43]:
# melihat 5 baris pertama, tanpa diinstal
!hdfs dfs -cat /user/vey/ecommerce/processed/ringkasan_kota_kategori.csv | head -5 

kota,kategori,total_pendapatan
Magelang,Elektronik,18775000
Magelang,Fashion,27750000
Magelang,Kesehatan & Kecantikan,17375000
Magelang,Makanan & Minuman,17525000


In [44]:
!hdfs fsck /user/vey/ecommerce/raw/transaksi_magelang.csv -files -blocks

Connecting to namenode via http://localhost:9870/fsck?ugi=vey&files=1&blocks=1&path=%2Fuser%2Fvey%2Fecommerce%2Fraw%2Ftransaksi_magelang.csv
FSCK started by vey (auth:SIMPLE) from /127.0.0.1 for path /user/vey/ecommerce/raw/transaksi_magelang.csv at Wed Sep 09 07:23:37 WIB 2026

/user/vey/ecommerce/raw/transaksi_magelang.csv 12329 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1858389489-127.0.1.1-1788410591372:blk_1073741833_1009 len=12329 Live_repl=1


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	12329 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 12329 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued for replication:	0

Erasu

In [46]:
# melihat keseluruhan status dalam cluster HDFS
!hdfs dfsadmin -report

Configured Capacity: 52518420480 (48.91 GB)
Present Capacity: 20851437568 (19.42 GB)
DFS Remaining: 20851273728 (19.42 GB)
DFS Used: 163840 (160 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: vey
Decommission Status : Normal
Configured Capacity: 52518420480 (48.91 GB)
DFS Used: 163840 (160 KB)
Non DFS Used: 28966010880 (26.98 GB)
DFS Remaining: 20851273728 (19.42 GB)
DFS Used%: 0.00%
DFS Remaining%: 39.70%
Configured Cache Capacity: 0 (0 B)
Cache Used: 0 

In [45]:
!hdfs dfs -stat "Ukuran: %b bytes Replikasi: %r Terakhir diubah: %y" /user/vey/ecommerce/raw/transaksi_magelang.csv

Ukuran: 12329 bytes Replikasi: 1 Terakhir diubah: 2026-09-08 14:31:45


In [47]:
# menampilkan block size
!hdfs dfs -stat "Ukuran: %b bytes Replikasi: %r Block Size: %o bytes Terakhir diubah: %y" /user/vey/ecommerce/raw/transaksi_magelang.csv

Ukuran: 12329 bytes Replikasi: 1 Block Size: 134217728 bytes Terakhir diubah: 2026-09-08 14:31:45


In [52]:
# melihat jumlah transaksi tiap kota
df_gabungan["kota"].value_counts()

kota
Magelang      200
Semarang      200
Yogyakarta    200
Name: count, dtype: int64

In [53]:
# melihat jumlah pendapatan perkota
df_gabungan.groupby("kota")["total_pendapatan"].sum()

kota
Magelang      93625000
Semarang      92100000
Yogyakarta    97275000
Name: total_pendapatan, dtype: int64

In [54]:
# cek kapasitas HDFS
!hdfs dfs -df -h

Filesystem               Size   Used  Available  Use%
hdfs://localhost:9000  48.9 G  160 K     19.4 G    0%


In [55]:
# Cek block data transaksi Magelang
!hdfs fsck /user/vey/ecommerce/raw/transaksi_magelang.csv -files -blocks

Connecting to namenode via http://localhost:9870/fsck?ugi=vey&files=1&blocks=1&path=%2Fuser%2Fvey%2Fecommerce%2Fraw%2Ftransaksi_magelang.csv
FSCK started by vey (auth:SIMPLE) from /127.0.0.1 for path /user/vey/ecommerce/raw/transaksi_magelang.csv at Wed Sep 09 14:11:28 WIB 2026

/user/vey/ecommerce/raw/transaksi_magelang.csv 12329 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1858389489-127.0.1.1-1788410591372:blk_1073741833_1009 len=12329 Live_repl=1


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	12329 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 12329 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued for replication:	0

Erasu